In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

In [3]:
df = pd.read_csv("googleplaystore.csv")
reviews_df = pd.read_csv("googleplaystore_user_reviews.csv")

print("Apps Shape:", df.shape)
print("Reviews Shape:", reviews_df.shape)
print(df.head())
print(reviews_df.head())

Apps Shape: (10841, 13)
Reviews Shape: (64295, 5)
                                                 App        Category  Rating  \
0     Photo Editor & Candy Camera & Grid & ScrapBook  ART_AND_DESIGN     4.1   
1                                Coloring book moana  ART_AND_DESIGN     3.9   
2  U Launcher Lite – FREE Live Cool Themes, Hide ...  ART_AND_DESIGN     4.7   
3                              Sketch - Draw & Paint  ART_AND_DESIGN     4.5   
4              Pixel Draw - Number Art Coloring Book  ART_AND_DESIGN     4.3   

  Reviews  Size     Installs  Type Price Content Rating  \
0     159   19M      10,000+  Free     0       Everyone   
1     967   14M     500,000+  Free     0       Everyone   
2   87510  8.7M   5,000,000+  Free     0       Everyone   
3  215644   25M  50,000,000+  Free     0           Teen   
4     967  2.8M     100,000+  Free     0       Everyone   

                      Genres      Last Updated         Current Ver  \
0               Art & Design   January 7, 20

In [4]:

sent_agg = reviews_df.groupby("App")[["Sentiment_Polarity", "Sentiment_Subjectivity"]].mean().reset_index()
sent_agg.columns = ["App", "Avg_Sentiment_Polarity", "Avg_Sentiment_Subjectivity"]

df = df.merge(sent_agg, on="App", how="left")


df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce")
df = df.sort_values("Last Updated", ascending=False).drop_duplicates(subset=["App"], keep="first")


df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df.dropna(subset=["Rating"])


df["Success"] = (df["Rating"] >= 4.0).astype(int)

print("\nTarget distribution (Success):")
print(df["Success"].value_counts(normalize=True))


Target distribution (Success):
Success
1    0.76711
0    0.23289
Name: proportion, dtype: float64


In [5]:
def clean_and_engineer_features(df_):
    df_ = df_.copy()

    # Reviews
    df_["Reviews"] = pd.to_numeric(df_["Reviews"], errors="coerce")

    # Installs: remove '+' and ',' -> numeric
    df_["Installs"] = (
        df_["Installs"]
        .astype(str)
        .str.replace("+", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    df_["Installs"] = pd.to_numeric(df_["Installs"], errors="coerce")

    # Price: remove '$'
    df_["Price"] = df_["Price"].astype(str).str.replace("$", "", regex=False)
    df_["Price"] = pd.to_numeric(df_["Price"], errors="coerce")

    # Size: remove 'M' / 'k' and handle 'Varies with device'
    size = df_["Size"].astype(str)
    size = size.str.replace("M", "", regex=False)
    size = size.str.replace("k", "", regex=False)
    size = size.replace("Varies with device", np.nan)
    df_["Size"] = pd.to_numeric(size, errors="coerce")

    # Review rate: Reviews per install
    df_["Review_Rate"] = df_["Reviews"] / (df_["Installs"] + 1)

    # Recency (days since last update)
    TODAY = pd.to_datetime("2018-08-01")
    df_["Last Updated"] = pd.to_datetime(df_["Last Updated"], errors="coerce")
    df_["Recency_Days"] = (TODAY - df_["Last Updated"]).dt.days

    # Price category (categorical feature)
    df_["Price_Category"] = pd.cut(
        df_["Price"].fillna(0),
        bins=[-0.01, 0, 1, 5, 20, np.inf],
        labels=["free", "cheap", "low", "medium", "high"]
    )

    return df_

df = clean_and_engineer_features(df)

In [6]:

drop_cols = ["App", "Rating", "Success", "Current Ver", "Android Ver", "Last Updated"]
drop_cols = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=drop_cols)
y = df["Success"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)



Train shape: (6557, 13)
Test shape: (1640, 13)


In [7]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


Numeric features: ['Reviews', 'Size', 'Installs', 'Price', 'Avg_Sentiment_Polarity', 'Avg_Sentiment_Subjectivity', 'Review_Rate', 'Recency_Days']
Categorical features: ['Category', 'Type', 'Content Rating', 'Genres', 'Price_Category']


In [9]:
K = 75

anova_selector = SelectKBest(score_func=f_classif, k=K)

In [10]:
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "SGDClassifier": SGDClassifier(
        loss="log_loss", max_iter=2000, alpha=1e-4, random_state=42
    ),
    "LinearSVC": LinearSVC(
        random_state=42
    ),
}

def make_baseline_pipeline(clf):
    """Pipeline بدون Feature Selection (فقط preprocessing + model)."""
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("clf", clf),
        ]
    )

def make_anova_pipeline(clf):
    """Pipeline مع SelectKBest(ANOVA)."""
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("select", anova_selector),
            ("clf", clf),
        ]
    )

def evaluate_model(pipeline, X_tr, X_te, y_tr, y_te, model_name, variant):
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)


    if hasattr(pipeline, "predict_proba"):
        y_score = pipeline.predict_proba(X_te)[:, 1]
    elif hasattr(pipeline, "decision_function"):
        y_score = pipeline.decision_function(X_te)
    else:
        y_score = None

    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec = recall_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    if y_score is not None:
        roc = roc_auc_score(y_te, y_score)
    else:
        roc = np.nan

    print(f"\n=== {model_name} ({variant}) ===")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1       :", f1)
    if not np.isnan(roc):
        print("ROC AUC  :", roc)
    print("\nClassification report:")
    print(classification_report(y_te, y_pred))

    return {
        "Model": model_name,
        "Variant": variant,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": roc,
    }

results = []

for name, clf in models.items():

    baseline_pipe = make_baseline_pipeline(clf)
    res_base = evaluate_model(
        baseline_pipe, X_train, X_test, y_train, y_test, name, "No_FS"
    )
    results.append(res_base)


    anova_pipe = make_anova_pipeline(clf)
    res_anova = evaluate_model(
        anova_pipe, X_train, X_test, y_train, y_test, name, f"ANOVA_k={K}"
    )
    results.append(res_anova)


=== LogisticRegression (No_FS) ===
Accuracy : 0.6359756097560976
Precision: 0.8874560375146542
Recall   : 0.6017488076311606
F1       : 0.7171956418758882
ROC AUC  : 0.7306349312046878

Classification report:
              precision    recall  f1-score   support

           0       0.36      0.75      0.49       382
           1       0.89      0.60      0.72      1258

    accuracy                           0.64      1640
   macro avg       0.63      0.68      0.60      1640
weighted avg       0.77      0.64      0.66      1640


=== LogisticRegression (ANOVA_k=75) ===
Accuracy : 0.626219512195122
Precision: 0.8880866425992779
Recall   : 0.5866454689984102
F1       : 0.7065581617999043
ROC AUC  : 0.7262545884350629

Classification report:
              precision    recall  f1-score   support

           0       0.36      0.76      0.49       382
           1       0.89      0.59      0.71      1258

    accuracy                           0.63      1640
   macro avg       0.62      0.

In [11]:
results_df = pd.DataFrame(results)
print("\n\n===== Summary Results  =====")
print(results_df.sort_values(by="F1", ascending=False))



===== Summary Results (Alaa) =====
                Model     Variant  Accuracy  Precision    Recall        F1  \
4           LinearSVC       No_FS  0.771341   0.775421  0.988076  0.868927   
3       SGDClassifier  ANOVA_k=75  0.770122   0.773433  0.990461  0.868595   
5           LinearSVC  ANOVA_k=75  0.769512   0.774314  0.987281  0.867925   
2       SGDClassifier       No_FS  0.768293   0.772333  0.989666  0.867596   
0  LogisticRegression       No_FS  0.635976   0.887456  0.601749  0.717196   
1  LogisticRegression  ANOVA_k=75  0.626220   0.888087  0.586645  0.706558   

    ROC_AUC  
4  0.725285  
3  0.722018  
5  0.721874  
2  0.723283  
0  0.730635  
1  0.726255  
